# Practice 2.3 · Equivocation 破除

> **对应知识点**：EP2 · KP3 · 强制做选择
> **测什么**：三招破 hedging，让模型给"一个明确答案"而不是"列排行榜"
> **预计时间**：5 分钟

---

## 你会看到什么

用一个模型默认会含糊的问题（"史上最伟大 NBA 球员是谁"），测 3 版 prompt 的效果差异：

- 版本 A · 直接问 → 模型给排行榜
- 版本 B · 加 "ONLY the name, no other words"
- 版本 C · B + fallback hedge（"如果必须犹豫也只能选一个"）

**测输出长度 + 是否包含 hedging 词**。


In [ ]:
# ============ 前置：安装 SDK + 配置 client ============
!pip install openai -q

from openai import OpenAI
from google.colab import userdata   # Colab；Modelscope 上用 os.environ 或直接填 key

# 推荐用 Secrets 存 key（不要硬编码）
API_KEY = userdata.get("DEEPSEEK_KEY")   # Colab Secrets
# 或 Modelscope 上：
# API_KEY = "sk-你的DeepSeek key"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com/v1"
)

MODEL = "deepseek-chat"

def call(prompt, **kwargs):
    """简化的 chat 调用。默认 temperature=0 保证实验可重现。"""
    kwargs.setdefault("temperature", 0)
    kwargs.setdefault("max_tokens", 512)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs
    )
    return r.choices[0].message.content


## 3 版 Prompt

In [ ]:
QUESTION = "史上最伟大的 NBA 球员是谁？"

# 版本 A · 直接问
PROMPT_A = QUESTION

# 版本 B · 强制格式
PROMPT_B = f"""{QUESTION}

只回答一个人名。不要解释、不要其他词、不要标点。"""

# 版本 C · B + fallback hedge
PROMPT_C = f"""{QUESTION}

只回答一个人名。不要解释、不要其他词、不要标点。
如果必须犹豫，也只能选最有代表性的那一个。"""

## 跑测试

In [ ]:
HEDGING_WORDS = ["取决于", "各有优势", "可以考虑", "看情况", "很难", "这是有争议", "多个", "候选", "或者"]

def has_hedging(text):
    return any(w in text for w in HEDGING_WORDS)

for label, prompt in [("A · 直接问", PROMPT_A), ("B · 强制格式", PROMPT_B), ("C · B+fallback", PROMPT_C)]:
    out = call(prompt, max_tokens=200)
    print(f"=== 版本 {label} ===")
    print(f"输出长度：{len(out)} 字符")
    print(f"包含 hedging 词：{'是' if has_hedging(out) else '否'}")
    print(f"输出：{out[:200]}")
    print()

# 手动判断：版本 C 应该最短、无 hedging、给单一名字

## 边界提醒

**低风险场景**（书单推荐 / 电影推荐 / 技术选型）→ 大胆用强制做选择。

**高风险场景**（医疗诊断 / 法律建议 / 重大人生决策）→ **不应该**强制。模型给排行榜是**负责任**的行为——这是 EP2 KP3 的关键边界。

## 一句心法

> **强制做选择让 chatbot 有"果断感"** —— 用户想要"你告诉我该选 A 还是 B"，不是"两个都不错，看你自己"。
